# L3c: Binary Classification and Regularization
In this lecture, we will explore logistic regression, a technique for binary classification tasks. We will also discuss the concept of regularization, which helps prevent overfitting in our models.

> __Learning Objectives:__
> 
> By the end of this lecture, you should be able to:
>
> * __Understand binary classification methods:__ Explain how the Perceptron learns a linear decision boundary using the sign function and contrast it with logistic regression's probabilistic approach using the logistic function.
> * __Derive logistic regression from the Boltzmann distribution:__ Represent binary classification as an energy-based model, derive the logistic function and cross-entropy loss from maximum likelihood estimation, and understand how the inverse temperature parameter controls prediction confidence.
> * __Apply optimization and regularization techniques:__ Use gradient descent to minimize cross-entropy loss for parameter learning and apply L1 and L2 regularization to prevent overfitting and improve model generalization.

Let's get started!
___

## Example
Today, we will use the following examples to illustrate key concepts:

> [▶ Build a Perceptron classifier of a banknote dataset](CHEME-5800-L9a-Example-LinearModels-Classification-Perceptron-Fall-2025.ipynb). Implement the Perceptron algorithm to classify authentic and inauthentic banknotes based on features extracted from images of the banknotes. This exercise demonstrates how to apply linear models to classification tasks.
 
> [▶ Logistic classification of a banknote dataset](CHEME-5820-L3c-Example-LogisticRegression-GD-Spring-2026.ipynb). In this example, we'll use logistic regression to classify authentic and inauthentic banknotes based on features extracted from images of the banknotes. We'll train a logistic regression model using gradient descent and evaluate its performance using the confusion matrix.

___

<div>
    <center>
        <img src="figs/Fig-LinearlySeparable-Schematic.svg" width="480"/>
    </center>
</div>

## Binary Classification Problem
Linear regression can be adapted for classification by transforming the continuous output into a class designation. We can do this in two ways: directly to a class designation or to a probability using an __activation function__ $\sigma:\mathbb{R}\rightarrow{\mathbb{R}}$.

Let's examine two binary classification strategies:

* [The Perceptron (Rosenblatt, 1957)](https://en.wikipedia.org/wiki/Perceptron) is an algorithm for binary classification that learns a linear decision boundary separating two classes. The Perceptron maps continuous output to a class such as $\sigma:\mathbb{R}\rightarrow\{-1,+1\}$ using $\sigma(\star) = \text{sign}(\star)$.
* [Logistic regression](https://en.wikipedia.org/wiki/Logistic_regression#) uses the [logistic function](https://en.wikipedia.org/wiki/Logistic_function) to transform linear regression output into a probability. This approach is effective for various applications. We'll explore logistic regression in the next module.

Today, we focus on __the Perceptron__ and __logistic regression__ as foundational methods for binary classification. The Perceptron provides a hard decision rule (class or no class), while logistic regression offers a probabilistic framework that quantifies confidence in predictions.
___

## Perceptron
[The Perceptron (Rosenblatt, 1957)](https://en.wikipedia.org/wiki/Perceptron) transforms the output of a linear regression model $y_{i}\in\mathbb{R}$ using an activation function to produce discrete class values. For binary classification, we use $\sigma:\mathbb{R}\rightarrow\{-1,1\}$. 

Suppose we have a labeled dataset $\mathcal{D} = \left\{(\mathbf{x}_{1},y_{1}),\dotsc,(\mathbf{x}_{n},y_{n})\right\}$ with $n$ examples, where each example is labeled by an expert in a category $y_{i}\in\{-1,1\}$ with an $m$-dimensional feature vector $\mathbf{x}_{i}\in\mathbb{R}^{m}$. 

> __Perceptron__
> 
> [The Perceptron](https://en.wikipedia.org/wiki/Perceptron) learns a linear decision boundary between two classes by repeatedly processing the data. During each pass, the parameter vector $\mathbf{\theta}$ is updated until the number of misclassifications meets a specified threshold. [The Perceptron](https://en.wikipedia.org/wiki/Perceptron) computes the estimated label $\hat{y}_{i}$ for feature vector $\hat{\mathbf{x}}_{i}$ using the $\texttt{sign}:\mathbb{R}\to\{-1,1\}$ function:
> $$
\begin{align*}
    \hat{y}_{i} &= \texttt{sign}\underbrace{\left(\hat{\mathbf{x}}_{i}^{\top}\;\mathbf{\theta}\right)}_{\left<\hat{\mathbf{x}}_{i}, \mathbf{\theta}\right>}
\end{align*}
$$
> where $\mathbf{\theta}=\left(w_{1},\dots,w_{m}, b\right)$ is a column vector of classifier parameters, with $w_{j}\in\mathbb{R}$ representing the importance of feature $j$ and $b\in\mathbb{R}$ a bias term. The augmented features $\hat{\mathbf{x}}^{\top}_{i}=\left(x^{(i)}_{1},\dots,x^{(i)}_{m}, 1\right)$ are $p = m+1$-dimensional vectors, and $\texttt{sign}(z)$ is defined as:
> $$
\begin{equation*}
    \texttt{sign}(z) = 
    \begin{cases}
        1 & \text{if}~z\geq{0}\\
        -1 & \text{if}~z<0
    \end{cases}
\end{equation*}
$$

### Classical: Online Perceptron Training
__Convergence guarantee__: If dataset $\mathcal{D}$ is linearly separable, the Perceptron converges to a separating hyperplane in finite iterations. However, if $\mathcal{D}$ is not linearly separable, the Perceptron may not converge. 

Let's examine the Perceptron learning algorithm pseudocode. 

__Initialize__: Given a linearly separable dataset $\mathcal{D} = \left\{(\mathbf{x}_{1},y_{1}),\dotsc,(\mathbf{x}_{n},y_{n})\right\}$, the maximum number of iterations $T$, and the maximum number of mistakes $M$ (e.g., $M=1$), initialize the parameter vector $\mathbf{\theta} = \left(\mathbf{w}, b\right)$ to small random values and set the loop counter $t\gets{0}$.

> **Rule of thumb for $T$**: Set $T = 10n$ to $100n$, where $n$ is the number of training examples. The algorithm often converges faster for linearly separable data.

While $\texttt{true}$ __do__:
1. Initialize the number of mistakes $\texttt{mistakes} = 0$.
2. For each training example $(\mathbf{x}, y) \in \mathcal{D}$: compute $y\;\left(\mathbf{\theta}^{\top}\;\mathbf{x}\right)\leq{0}$. 
    - If true: the example is __misclassified__ (the sign of the prediction doesn't match the label $y$). Update $\mathbf{\theta} \gets \mathbf{\theta} + y\;\mathbf{x}$ and increment $\texttt{mistakes} \gets \texttt{mistakes} + 1$.
3. After processing all examples, if $\texttt{mistakes} \leq {M}$ or $t \geq T$, exit. Otherwise, increment $t \gets t + 1$ and repeat from step 1.

We aim to minimize mistakes, with $M = 0$ being ideal. However, zero mistakes may not be achievable for weakly separable or non-separable data.

Let's look at example of the perceptron algorithm in action!

> [▶ Build a Perceptron classifier of a banknote dataset](CHEME-5800-L9a-Example-LinearModels-Classification-Perceptron-Fall-2025.ipynb). Implement the Perceptron algorithm to classify authentic and inauthentic banknotes based on features extracted from images of the banknotes. This exercise demonstrates how to apply linear models to classification tasks.

___

## Logistic Regression: Cross-entropy loss
Unlike the Perceptron's deterministic approach, we now adopt a probabilistic framework for binary classification. This allows us to not only predict class labels but also quantify confidence in our predictions.

Suppose we view our two–class labels $y\in\{-1,1\}$ as _states_ in a Boltzmann distribution conditioned on the input $\hat{\mathbf{x}}\in\mathbb{R}^{m+1}$ (the original feature vector with a `1` as the last element to account for a bias). Then for any state $y$ with energy $E(y,\hat{\mathbf{x}})$ at inverse temperature $\beta$, the conditional probability of observing the label $y\in\left\{-1,+1\right\}$ given the feature vector $\hat{\mathbf{x}}$ can be represented as
$$
\begin{align*}
P(y\mid \hat{\mathbf{x}})
=\frac{\exp\bigl(-\beta E(y,\hat{\mathbf{x}})\bigr)}
      {\underbrace{\sum_{y' \in\{-1,1\}} \exp\bigl(-\beta E(y',\hat{\mathbf{x}})\bigr)}_{Z(\hat{\mathbf{x}})}}.
\end{align*}
$$

The partition function $Z(\hat{\mathbf{x}}) = \sum_{y' \in\{-1,1\}} \exp(-\beta E(y',\hat{\mathbf{x}}))$ normalizes the probabilities so they sum to one across all possible labels.

### Understanding Inverse Temperature $\beta$

The inverse temperature $\beta > 0$ is a fundamental parameter that controls how the energy function translates into probabilities. Think of $\beta$ as controlling the "confidence" or "commitment" of the model to its decisions:

> __Physical Intuition for $\beta$:__
> 
> * **Large $\beta$ (high "temperature sharpness"):** The exponential function becomes increasingly selective, small energy differences translate into large probability differences. The model becomes highly confident, approaching a hard decision boundary. As $\beta \to \infty$, the probability distribution becomes sharper and sharper, eventually resembling the sign function of the Perceptron.
> * **Small $\beta$ (low "temperature sharpness"):** The exponential function flattens, and all states become roughly equally probable regardless of energy differences. The model becomes uncertain, assigning probability near 0.5 to both classes. As $\beta \to 0$, predictions approach maximum uncertainty (50/50 chance for either class).
> * **$\beta = 1$ (default):** Provides a balance between confidence and uncertainty, often used as the baseline.
> 
> In practice, $\beta$ is often absorbed into the weight vector $\theta$ (by rescaling), but keeping it explicit allows independent control of prediction confidence. In applications like medical diagnosis, you might use $\beta < 1$ to avoid overconfident predictions; in spam detection, $\beta > 1$ makes the model more decisive.

For the energy function, we use a linear model of the form:
$$
\begin{align*}
E(y,\hat{\mathbf{x}})\;=\;-\,y\;\bigl(\hat{\mathbf{x}}^{\top}\theta \bigr).
\end{align*}
$$
where $\theta\in\mathbb{R}^{p}$ is a vector of __unknown__ parameters (weights plus bias) that we want to learn. This energy definition creates a natural alignment: when $\hat{\mathbf{x}}^{\top}\theta$ is large and positive, the energy for $y=+1$ is low (favoring that label), while the energy for $y=-1$ is high (disfavoring it).

When $y=+1$, the energy $E(1,\hat{\mathbf{x}})=-\hat{\mathbf{x}}^{\top}\theta$ is *lower* (more probable) if $\hat{\mathbf{x}}^{\top}\theta$ is large. On the other hand, when $y=-1$, the energy $E(-1,\hat{\mathbf{x}})=+\hat{\mathbf{x}}^{\top}\theta$, so $y=-1$ is favored when $\hat{\mathbf{x}}^{\top}\theta$ is very negative.

Let's substitute the energy function into the conditional probability expression and do some algebra:
$$
\begin{align*}
P_{\theta}(y\mid \hat{\mathbf{x}})
& =\frac{\exp\bigl(-\beta E(y,\hat{\mathbf{x}})\bigr)}
      {\underbrace{\sum_{y' \in\{-1,1\}} \exp\bigl(-\beta E(y',\hat{\mathbf{x}})\bigr)}_{Z(\hat{\mathbf{x}})}}\\
&=\frac{\exp\bigl(\beta y\left(\hat{\mathbf{x}}^{\top}\theta\right)\bigr)}
      {\exp\bigl(\beta\hat{\mathbf{x}}^{\top}\theta\bigr) + \exp\bigl(-\beta\hat{\mathbf{x}}^{\top}\theta\bigr)}\quad\Longrightarrow\;{\text{substituting } z = \beta\hat{\mathbf{x}}^{\top}\theta}\\
& = \frac{\exp\bigl(yz\bigr)}
      {\exp\bigl(z\bigr) + \exp\bigl(-z\bigr)}\quad\Longrightarrow\;{\text{factor out}\; \exp(yz)\;\text{from denominator}}\\
& = \frac{\exp\bigl(yz\bigr)}
      {\exp\bigl(yz\bigr)\left(\exp\bigl((1-y)z\bigr) + \exp\bigl(-(1+y)z\bigr)\right)}\quad\Longrightarrow\;\text{cancel}\;\exp(yz)\\
& = \frac{1}
      {\exp\bigl((1-y)z\bigr) + \exp\bigl(-(1+y)z\bigr)}\quad\blacksquare\\
\end{align*}
$$

This expression is the probability of observing the label $y$ given the feature vector $\hat{\mathbf{x}}$ and the parameters $\theta$. Let's look at the case when $y=+1$ and $y=-1$:

> __Cases:__
>
> When $y=+1$, we have:
> $$
\begin{align*}
P_{\theta}(y = +1\mid \hat{\mathbf{x}})
& = \frac{1}
      {\exp\bigl(0\bigr) + \exp\bigl(-2\beta\left(\hat{\mathbf{x}}^{\top}\theta\right)\bigr)}\\
& = \frac{1}
      {1 + \exp\bigl(-2\beta\left(\hat{\mathbf{x}}^{\top}\theta\right)\bigr)}\quad\blacksquare\\
\end{align*}
$$
> 
> When $y=-1$, we have:
> $$\begin{align*}
P_{\theta}(y = -1\mid \hat{\mathbf{x}})
& = \frac{1}
      {\exp\bigl(2\beta\left(\hat{\mathbf{x}}^{\top}\theta\right)\bigr) + \exp\bigl(0\bigr)}\\
& = \frac{1}
      {1+\exp\bigl(2\beta\left(\hat{\mathbf{x}}^{\top}\theta\right)\bigr)}\quad\blacksquare\\
\end{align*}
$$
> Putting this all together, we can write the conditional probability of observing the label $y$ given the feature vector $\hat{\mathbf{x}}$ and the parameters $\theta$ as:
> $$\begin{align*}
P_{\theta}(y\mid \hat{\mathbf{x}}) & = \frac{1}{1+\exp\bigl(-2\beta y\left(\hat{\mathbf{x}}^{\top}\theta\right)\bigr)}\quad\Longrightarrow\;\text{Logistic function!}\\
& = \sigma\bigl(2\beta y\left(\hat{\mathbf{x}}^{\top}\theta\right)\bigr)\\
\end{align*}$$

The logistic function $\sigma(\cdot)$ is a sigmoid activation function that maps any real-valued input to the interval $(0, 1)$, making it ideal for modeling probabilities. In logistic regression, the function compresses the linear predictor $2\beta y(\hat{\mathbf{x}}^{\top}\theta)$ into a probability space, providing a smooth, differentiable decision boundary between classes.

### Edge Cases: Extreme Values of $\beta$

Understanding the behavior at extreme values of $\beta$ provides insight into when logistic regression approaches the Perceptron and when it becomes maximally uncertain:

> __Extreme Cases of $\beta$:__
> 
> The choice of $\beta$ trades off between model confidence and calibration. 
> * __High-confidence regime (Cool system):__
> As $\beta \to \infty$, the logistic function becomes arbitrarily sharp. For any input where $2y(\hat{\mathbf{x}}^{\top}\theta) > 0$, the probability approaches 1; where $2y(\hat{\mathbf{x}}^{\top}\theta) < 0$, it approaches 0. The smooth sigmoid becomes a hard threshold, the model transitions to deterministic decision-making identical to the Perceptron's sign function. This is useful when you want maximum confidence, but at the cost of losing probabilistic interpretation.
> 
> * __Low-confidence regime (Hot system):__
> As $\beta \to 0$, the logistic function approaches the constant 0.5 for all inputs (except at the exact decision boundary). The model becomes unable to discriminate between classes, treating all predictions as equally uncertain regardless of $\hat{\mathbf{x}}^{\top}\theta$. While this seems useless, it can be valuable during exploratory learning when you want to regularize away overconfident predictions.
> 
> __Practical Implications:__
> Well-calibrated probabilistic models often use $\beta$ values closer to 1, while risk-averse applications (e.g., disease screening) might use smaller $\beta$ to avoid false confidence, and high-stakes applications (e.g., fraud detection) might use larger $\beta$ for decisive action.

### Parameter Estimation: Learning from Data
Now that we have a probabilistic model, the logistic function $\sigma(2\beta y(\hat{\mathbf{x}}^{\top}\theta))$, we need to learn the parameters $\theta$ that make our model fit the observed data well. We do this by finding parameters that maximize the likelihood of observing the training labels given the features.

Of course, we want to learn the parameters $\theta$ so that we maximize the log likelihood (or minimize the negative log-likelihood) of the observed labels given the feature vectors. The likelihood function is given by:
$$
\begin{align*}
\mathcal{L}(\theta) & = \prod_{i=1}^{n} P_{\theta}(y_{i}\mid \hat{\mathbf{x}}_{i})\\
& = \prod_{i=1}^{n} \frac{1}{1+\exp\bigl(-2\beta y_{i}\,\left(\hat{\mathbf{x}}^{\top}_{i}\theta\right)\bigr)}\quad\Longrightarrow\;\text{Product is $\textbf{hard}$ to optimize! Take the $\log$}\\
\log\mathcal{L}(\theta) & = -\sum_{i=1}^n \log\!\bigl(1+\exp\bigl(-2\beta y_i\,\left(\hat{\mathbf{x}}^{\top}_{i}\theta\right)\bigr)\bigr)\\
\end{align*}
$$  

We can use gradient descent to minimize the negative log-likelihood (also known as the cross-entropy loss function):
$$
\boxed{
\begin{align*}
J(\theta) & = -\log\mathcal{L}(\theta)\\
& = \sum_{i=1}^n \log\!\bigl(1+\exp\bigl(-2\beta y_i\,\left(\hat{\mathbf{x}}^{\top}_{i}\theta\right)\bigr)\bigr)\quad\blacksquare\\
\end{align*}}
$$      
This will give us the optimal parameters $\theta$ for our logistic regression model:
$$
\hat{\theta} = \arg\min_{\theta} J(\theta)
$$

Gradient descent can minimize this loss function to learn the optimal parameters for binary classification. The example notebook demonstrates this approach on a real dataset.

### Advanced Review Topics: Optimization Methods
For deeper understanding of gradient descent and constrained optimization, see the advanced notebooks:

> * [▶ Gradient Descent](docs/CHEME-5820-L3c-Advanced-GradientDescent-Spring-2026.ipynb). This notebook develops gradient descent algorithms for both unconstrained and constrained optimization problems, including barrier and penalty methods for handling constraints.
> * [▶ General Nonlinear Optimization Problem](docs/CHEME-5820-L3c-Advanced-GeneralProblem-Spring-2026.ipynb). This notebook presents the general constrained nonlinear optimization framework, Lagrangian formulation, and Karush-Kuhn-Tucker (KKT) optimality conditions.

___

## Regularization: Preventing Overfitting
While gradient descent successfully minimizes the cross-entropy loss, the resulting model may fit the training data too closely, capturing noise and irrelevant patterns. This overfitting problem is especially severe in high-dimensional feature spaces or with limited training data. We address this through regularization.

In practice, logistic regression models can overfit to training data, particularly when the feature space is high-dimensional or when the training set is small. 

> __What is regularization?__
>
> __Regularization__ is a technique that adds a penalty term to the loss function to discourage complex models and improve generalization to unseen data.
> 
> By penalizing large parameter values, regularization encourages simpler, more generalizable decision boundaries. Two common regularization approaches are L2 (Ridge) and L1 (Lasso) regularization.
> 
> * __L2 regularization (Ridge):__ $R(\theta) = \frac{1}{2}\lVert\theta\rVert_{2}^{2} = \frac{1}{2}\sum_{j=1}^{p}\theta_{j}^{2}$. This penalizes the squared magnitude of all parameters equally and encourages smaller parameter values across the board.
> * __L1 regularization (Lasso):__ $R(\theta) = \lVert\theta\rVert_{1} = \sum_{j=1}^{p}|\theta_{j}|$. This penalizes the absolute magnitude of parameters and can drive some parameters exactly to zero, effectively performing automatic feature selection.

The regularized cross-entropy loss function can be written as:
$$
\boxed{
\begin{align*}
J_{\text{reg}}(\theta) & = J(\theta) + \lambda\,R(\theta)\\
& = \sum_{i=1}^n \log\!\bigl(1+\exp\bigl(-2\beta y_i\,\left(\hat{\mathbf{x}}^{\top}_{i}\theta\right)\bigr)\bigr) + \lambda\,R(\theta)\quad\blacksquare\\
\end{align*}}
$$

where $\lambda > 0$ is the __regularization parameter__ (also called the regularization strength) that controls the trade-off between minimizing training error and keeping parameters small, and $R(\theta)$ is the regularization function. 

> __Important: Feature Scaling and Regularization__
>
> * __Feature scale__ features to comparable magnitudes (e.g., zero mean, unit variance) before regularization so penalties are applied fairly across parameters.
> * __Regularization parameter:__ The regularization parameter $\lambda$ controls the strength of regularization: small values of $\lambda$ give more weight to the training loss, while large values emphasize parameter magnitude control. Selecting an appropriate $\lambda$ is crucial and is typically done via cross-validation.


Let's look at an example of logistic regression with regularization!

> [▶ Logistic classification of a banknote dataset](CHEME-5820-L3c-Example-LogisticRegression-GD-Spring-2026.ipynb). In this example, we'll use logistic regression to classify authentic and inauthentic banknotes based on features extracted from images of the banknotes. We'll train a logistic regression model using gradient descent and evaluate its performance using the confusion matrix.

___

## Summary
Logistic regression uses the Boltzmann distribution and cross-entropy loss to model binary classification problems and learn decision boundaries through maximum likelihood estimation.

> __Key Takeaways:__
>
> * **Binary classification spans deterministic to probabilistic approaches:** The Perceptron provides hard-threshold classification guaranteed to converge for linearly separable data, while logistic regression emerges from the Boltzmann distribution framework to yield a probabilistic decision function. The inverse temperature parameter controls model confidence, ranging from highly uncertain to nearly deterministic Perceptron-like behavior.
> * **Cross-entropy loss optimization learns classification boundaries:** The cross-entropy loss function arises naturally from maximum likelihood estimation and must be minimized using gradient descent to find optimal parameters that maximize the probability of observed training labels, making it the foundation for practical classifier training.
> * **Regularization prevents overfitting through parameter constraints:** Adding L1 penalties for automatic feature selection or L2 penalties for parameter shrinkage to the loss function controls model complexity. The regularization strength parameter balances training fit against generalization, and proper feature scaling is essential to ensure fair regularization across all parameters.


Logistic regression provides a practical and interpretable approach to binary classification by combining probabilistic modeling with gradient-based optimization.
___